# Séquence 4 — Qualité des données et incertitude
Parcours sans corrigé. À chaque étape: action, confiance, preuve, incertitude, limite.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from iot_decision.quality import load_raw,flatten,validate_row,classify,detect_gaps,diagnose
sample=ROOT/'data/samples/batch002_quality_messages.jsonl'

## Observer avant de nettoyer
Prédisez le nombre de messages et de zones avant de lire. Une valeur numérique présente n'est pas encore une valeur validée.

In [ ]:
envelopes=load_raw(sample)
rows=[flatten(e) for e in envelopes]
len(rows),sorted({r['zone'] for r in rows})

## Valider ligne à ligne
Quatre contrôles, dans cet ordre: champ manquant, unité incohérente, valeur hors plage physique, incohérence temporelle (`measured_at` postérieur à `received_at`). Aucune valeur n'est corrigée: une ligne est propre ou rejetée, avec une raison.

In [ ]:
clean,rejected=classify(rows)
len(clean),len(rejected),sorted(r['rejection_reason'] for r in rejected)

## Silence réel ou effet du filtrage ?
Une zone peut sembler avoir un trou dans sa chronologie simplement parce qu'une de ses lignes a été rejetée. Comparez chaque silence détecté à l'emplacement des lignes rejetées de la même zone avant de conclure à une vraie absence de message.

In [ ]:
gaps=detect_gaps(clean)
report=diagnose(rows,clean,rejected,gaps)
{zone:[(g['duration_minutes'],g['explained_by_rejection']) for g in zone_gaps] for zone,zone_gaps in report.gaps_by_zone.items()}

## Votre rapport qualité
Le silence de `battery-shelter-01` précède directement une valeur au-dessus du seuil pédagogique de 35 °C. Rédigez votre rapport qualité: quelle confiance accordez-vous à cette alerte, quelles preuves le justifient, quelles incertitudes subsistent, et quelle vérification proposez-vous avant toute action ? L'appel suivant teste seulement l'API.

In [ ]:
assert report.confidence in {'très faible','faible','moyenne','élevée'}
assert set(report.rejected_by_reason) <= {
    'champ manquant','unité incohérente','valeur hors plage physique',
    'incohérence temporelle','doublon exact',
}